# Twitter Crisis-Signal Analysis Pipeline (2020)
## Extended Research Framework

This notebook orchestrates the research pipeline for aggregate crisis-signal analysis, starting from raw tweet IDs and culminating in an algorithmic phase-segmentation dashboard.

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from glob import glob

# Import custom modules from src
from src.config import RAW_DIR, HYDRATED_DIR, PROCESSED_DIR, REPORTS_DIR, START_DATE, END_DATE
from src.hydration import extract_tweets_from_jsonl
from src.data_prep import preprocess_dataframe
from src.features import enrich_features
from src.embeddings import generate_embeddings
from src.aggregation import build_daily_sequence
from src.lstm_ae import create_sequences, train_lstm_autoencoder, extract_hidden_states
from src.segmentation import detect_changepoints, cluster_hidden_states, map_phases
from src.dashboard import create_research_dashboard, compute_baseline_mse, save_metrics

print(f"Pipeline initialized. Study Window: {START_DATE.date()} to {END_DATE.date()}")

### 1. Data Loading & Hydration
Assuming hydration is handled or we have hydrated CSVs in `data/hydrated`.

In [ ]:
hydrated_csvs = glob(os.path.join(HYDRATED_DIR, "*.csv"))
if not hydrated_csvs:
    print("No hydrated CSVs found. Please run src/hydration.py or provide files.")
else:
    print(f"Loading {len(hydrated_csvs)} files...")
    df_list = [pd.read_csv(f) for f in hydrated_csvs]
    df_raw = pd.concat(df_list, ignore_index=True)
    print(f"Total raw tweets loaded: {len(df_raw)}")

### 2. Preprocessing & Feature Enrichment
Applying dual-track cleaning and lexicon-based scoring (VADER, NRC, Keyword flags).

In [ ]:
print("Starting Preprocessing...")
df_clean = preprocess_dataframe(df_raw)

print("Enriching Features (Sentiment, Distress, Rumors)...")
df_enriched = enrich_features(df_clean)

display(df_enriched.head(3))

### 3. BERTweet Embeddings
Generating and caching 768-D embeddings.

In [ ]:
print("Generating BERTweet Embeddings (this may take time)...")
# file_prefix targets the save name for the .npy cache
generate_embeddings(df_enriched, file_prefix="study_dataset")

### 4. Daily Multivariate Aggregation

In [ ]:
daily_csv = os.path.join(PROCESSED_DIR, "daily_crisis_signals.csv")
df_daily = build_daily_sequence(df_enriched, daily_csv)
display(df_daily.head())
print(f"Aggregate sequence built with {len(df_daily)} days.")

### 5. Deep Learning: LSTM Autoencoder
Training to extract temporal bottleneck latent states.

In [ ]:
X = create_sequences(df_daily)
print(f"Sequential data shape: {X.shape}")

model, scaler = train_lstm_autoencoder(X, epochs=50, hidden_dim=32)

# Scaled full data for latent extraction
X_scaled = scaler.transform(X.reshape(-1, X.shape[2])).reshape(X.shape)
hidden_states = extract_hidden_states(model, X_scaled)
print(f"Extracted {len(hidden_states)} hidden states (one per sliding window).")

### 6. Phase Segmentation & Clustering

In [ ]:
cpts = detect_changepoints(hidden_states, pen=15)
labels, sil_score = cluster_hidden_states(hidden_states, n_clusters=3)

df_final = map_phases(df_daily, cpts, labels)
print("Mapping complete. Calculating reconstruction metrics...")

model.eval()
with torch.no_grad():
    recon = model(torch.tensor(X_scaled, dtype=torch.float32).to(next(model.parameters()).device)).cpu().numpy()
    ae_mse = np.mean((X_scaled - recon)**2)

baseline_mse = compute_baseline_mse(X_scaled)
print(f"LSTM AE MSE: {ae_mse:.5f} | Baseline MSE: {baseline_mse:.5f}")

### 7. Evaluation & Export

In [ ]:
dashboard_path = os.path.join(REPORTS_DIR, "crisis_signal_dashboard.png")
create_research_dashboard(df_final, hidden_states, ae_mse, baseline_mse, dashboard_path)

metrics = {
    "ae_mse": float(ae_mse),
    "baseline_mse": float(baseline_mse),
    "silhouette_score": float(sil_score),
    "changepoint_indices": [int(c) for c in cpts]
}
save_metrics(metrics, os.path.join(REPORTS_DIR, "metrics.json"))

print("--- Run Complete ---")